# 18 — 365scores trends per-match snapshot

Hourly fetcher for the data-driven trends 365scores surfaces in their match cards. Feeds A12 / A13 in the recommender factor catalog. Each row carries a `snapshot_ts` so we can study how a trend's `percentage` evolves through the days before kickoff. After a match completes 365 publishes `outcome ∈ {1=hit, 2=miss}` per trend — the gold for EDA hit-rate calibration.

**Inputs**: `wc26_stg_matches.parquet` (for espn_match_id + nation pair).
**Output**: `wc26_match_trends_365.parquet` — long format, one row per (espn_match_id, trend_id, snapshot_ts).

Mirrors the PWA's `src/services/scores365.ts` — same endpoint, same dedup logic, same competitor-name matcher.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if (ROOT / "lib").is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "lib").is_dir():
    sys.path.insert(0, str(ROOT.parent))

from lib import scores365
from lib import io as wio

## 1. Build the join key (espn_match_id + nation names)

In [2]:
matches = pd.read_parquet(ROOT / "data/processed/wc26_stg_matches.parquet")
nations = pd.read_parquet(ROOT / "data/processed/wc26_stg_nations.parquet")[["nation_id", "seed_name"]]

key = matches[["espn_match_id", "home_nation_id", "away_nation_id", "stage", "kickoff_utc", "status"]].copy()
key = key.merge(nations.rename(columns={"nation_id": "home_nation_id", "seed_name": "home_nation_name"}), on="home_nation_id", how="left")
key = key.merge(nations.rename(columns={"nation_id": "away_nation_id", "seed_name": "away_nation_name"}), on="away_nation_id", how="left")

# Drop KO TBDs — 365 won't have trends for unfilled bracket slots
key = key.dropna(subset=["home_nation_name", "away_nation_name"])
print(f"resolvable fixtures: {len(key)}")
key.head()

resolvable fixtures: 72


,espn_match_id,home_nation_id,away_nation_id,stage,kickoff_utc,status,home_nation_name,away_nation_name
0,760415,MEX,RSA,group_a,2026-06-11 19:00:00+00:00,finished,Mexico,South Africa
1,760414,KOR,CZE,group_a,2026-06-12 02:00:00+00:00,finished,South Korea,Czechia
2,760416,CAN,BIH,group_b,2026-06-12 19:00:00+00:00,finished,Canada,Bosnia & Herzegovina
3,760417,USA,PAR,group_d,2026-06-13 01:00:00+00:00,finished,United States,Paraguay
4,760420,QAT,SUI,group_b,2026-06-13 19:00:00+00:00,finished,Qatar,Switzerland


## 2. Fetch trends per game (single tick = one snapshot)

In [3]:
trends_df = scores365.build(key)
print(f"rows: {len(trends_df)}")
print(f"fixtures covered: {trends_df['espn_match_id'].nunique()}")
print(f"unique trend_ids: {trends_df['trend_id'].nunique()}")
trends_df.head()

  discovered 72 365scores games


rows: 459
fixtures covered: 69
unique trend_ids: 459


,espn_match_id,scores365_game_id,trend_id,snapshot_ts,text,cause,betCTA,percentage,lineTypeId,isTop,outcome,competitorIds,confidenceTrendIds
0,760415,4627866,11584516,2026-06-24T12:12:21.502598+00:00,Mexico won - 6/8 Last Matches,Mexico won,Mexico to win,0.750000,1,False,1.0,[5106],[12053815]
1,760415,4627866,12053815,2026-06-24T12:12:21.502598+00:00,Mexico won at home - 4/6 Last Matches,Mexico won at home,Mexico to win,0.666667,1,False,1.0,[5106],[11584516]
2,760415,4627866,11333632,2026-06-24T12:12:21.502598+00:00,Under 2.5 Goals - 4/5 Last Matches,Under 2.5 Goals,Under,0.800000,3,False,1.0,[5106],[]
3,760415,4627866,11584517,2026-06-24T12:12:21.502598+00:00,Mexico scored first - 6/8 Last Matches,Mexico scored first,Mexico to score first,0.750000,7,False,1.0,[5106],[]
4,760415,4627866,12004039,2026-06-24T12:12:21.502598+00:00,Mexico won the first half - 5/6 Last Matches,Mexico won the first half,Mexico to win 1st half,0.833333,5,False,1.0,[5106],[]


## 3. Append to history (don't overwrite — we want the snapshot timeline)

In [4]:
dst = ROOT / "data/processed/wc26_match_trends_365.parquet"
if dst.exists():
    prior = pd.read_parquet(dst)
    combined = pd.concat([prior, trends_df], ignore_index=True)
    # Dedup on the natural key — keep latest snapshot per (match, trend, ts)
    combined = combined.drop_duplicates(
        subset=["espn_match_id", "trend_id", "snapshot_ts"], keep="last"
    )
else:
    combined = trends_df

combined.to_parquet(dst, index=False)
size_kb = dst.stat().st_size / 1024
print(f"wrote {dst.name}  ({len(combined)} cumulative rows, {size_kb:.1f} KB)")

wrote wc26_match_trends_365.parquet  (459 cumulative rows, 22.8 KB)


## 4. Verification

- `isTop=True` should appear at most once per (espn_match_id, snapshot_ts).
- `outcome` should be null for upcoming fixtures, 1 or 2 for completed.
- `lineTypeId` distribution tells us category mix.

In [5]:
print('top trend count per (match, snapshot):')
top_per = trends_df[trends_df['isTop']].groupby(['espn_match_id', 'snapshot_ts']).size()
print(f'  max: {top_per.max() if len(top_per) else 0} (should be 1)')
print()
print('outcome distribution:')
print(trends_df['outcome'].value_counts(dropna=False))
print()
print('lineTypeId distribution:')
print(trends_df['lineTypeId'].value_counts())

top trend count per (match, snapshot):
  max: 3 (should be 1)

outcome distribution:
outcome
1.0    180
NaN    168
2.0    111
Name: count, dtype: int64

lineTypeId distribution:
lineTypeId
1     126
3     105
7      76
14     73
12     47
5      32
Name: count, dtype: int64
